# Basic Neural Simulation with NeuroWorkflow

This notebook demonstrates how to create a simple neural simulation workflow using the NeuroWorkflow library. We'll build a workflow that:

1. Creates a neural network from SONATA format
2. Simulates the network using NEST

## Setup

First, let's make sure we can import the NeuroWorkflow library. If you've installed it with pip, you can import it directly. Otherwise, we'll add the source directory to the Python path.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add the src directory to the Python path if needed
src_path = os.path.abspath(os.path.join(os.getcwd(), '../src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import NeuroWorkflow components
from neuroworkflow import WorkflowBuilder
from neuroworkflow.nodes.network.BuildSonataNetworkNode import BuildSonataNetworkNode
from neuroworkflow.nodes.simulation.SimulateSonataNetworkNode import SimulateSonataNetworkNode

## Creating Nodes

Now, let's create the nodes for our workflow. We'll need:

1. A network builder node to create the neural network
2. A simulation node to run the simulation

In [ ]:
# Create a network builder node
build_network = BuildSonataNetworkNode("SonataNetworkBuilder")

# Configure the network builder
build_network.configure(
    sonata_path="../data/300_pointneurons",  # Path to our SONATA configuration
    net_config_file="circuit_config.json",
    sim_config_file="simulation_config.json",
    hdf5_hyperslab_size=1024
)

# Print node information
print(build_network)

In [ ]:
# Create a simulation node
simulate_network = SimulateSonataNetworkNode("SonataNetworkSimulation")

# Configure the simulation
simulate_network.configure(
    simulation_time=1000.0,  # Simulation time in milliseconds
    record_from_population="internal",
    record_n_neurons=40
)

# Print node information
print(simulate_network)

## Building the Workflow

Now that we have our nodes, let's build a workflow that connects them together.

In [ ]:
# Create a workflow builder
workflow_builder = WorkflowBuilder("neural_simulation")

# Add nodes to the workflow
workflow_builder.add_node(build_network)
workflow_builder.add_node(simulate_network)

# Connect the nodes
# The output 'sonata_net' from the builder node connects to the input 'sonata_net' of the simulation node
workflow_builder.connect(
    "SonataNetworkBuilder", "sonata_net", 
    "SonataNetworkSimulation", "sonata_net"
)

# Connect node collections
workflow_builder.connect(
    "SonataNetworkBuilder", "node_collections", 
    "SonataNetworkSimulation", "node_collections"
)

# Build the workflow
workflow = workflow_builder.build()

# Print workflow information
print(workflow)

## Executing the Workflow

Now let's execute the workflow and see the results.

In [ ]:
# Execute the workflow
print("Executing workflow...")
success = workflow.execute()

if success:
    print("\nWorkflow execution completed successfully!")
else:
    print("\nWorkflow execution failed!")

## Accessing Results

After the workflow has executed, we can access the results from the output ports of the nodes.

In [ ]:
# Get the spike recorder from the simulation node
spike_recorder = simulate_network.get_output_port("spike_recorder").value

# Print information about the spike recorder
print("Spike Recorder Information:")

# Check if we have spike events
print(f"\nRecorded {len(spike_recorder.events['times'])} spikes")
print(f"From {len(set(spike_recorder.events['senders']))} unique neurons")
print(f"Neuron IDs: {spike_recorder.events['senders']}")


## Visualizing the Workflow

Let's create a simple visualization of our workflow using NetworkX and Matplotlib.

In [ ]:
try:
    import networkx as nx
    
    # Create a directed graph
    G = nx.DiGraph()
    
    # Add nodes
    for node_name in workflow.nodes:
        G.add_node(node_name)
    
    # Add edges for connections
    for conn in workflow.connections:
        G.add_edge(
            conn.from_node, 
            conn.to_node, 
            label=f"{conn.from_port} → {conn.to_port}"
        )
    
    # Create the plot
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G, seed=42)  # positions for all nodes
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=3000, node_color="lightblue")
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, width=2, arrowsize=20)
    
    # Draw node labels
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold")
    
    # Draw edge labels
    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)
    
    plt.title("Neural Simulation Workflow")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("NetworkX not installed. Install with: pip install networkx")

## Conclusion

In this notebook, we've demonstrated how to:

1. Create nodes for network building and simulation
2. Configure the nodes with appropriate parameters
3. Build a workflow by connecting the nodes
4. Execute the workflow to simulate a neural network
5. Access the simulation results including spike data
6. Visualize the spike trains and firing rates
7. Visualize the workflow structure

This example shows the basic structure of a NeuroWorkflow for neural simulation. The workflow:

- Loads SONATA configuration files from the 300_pointneurons directory
- Builds a network with excitatory and inhibitory populations
- Simulates the network for a specified time period
- Records and visualizes spike activity

In more complex scenarios, you can add additional nodes for data loading, analysis, visualization, and more, as demonstrated in the other notebooks in this series.